In [65]:
from pathlib import Path

from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import cohere
import pandas as pd
import os

load_dotenv()

True

Read items dataset

In [66]:
df_items=pd.read_json("../../data/meta_Appliances_with_category_ratings_100.jsonl",lines=True)

In [67]:
df_items

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Amazon Home,HERISUN 100 Disposable Coffee Filters for Sing...,4.4,360,[【Good Fits in All Refillable Pods】K cup paper...,[],7.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],HERISUN,"[Small Appliance Parts & Accessories, Coffee &...",{'Product Dimensions': '4.7 x 2.5 x 2.5 inches...,B0BJKNMKRP,NaN
1,Tools & Home Improvement,Large Capacity Egg Holder For Refrigerator - 3...,4.6,1285,[Auto Rolling Egg Holder: Egg container keeps ...,[Large Capacity Egg Container For Refrigerator...,24.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Honest review of egg holder for fr...,TOGOO,"[Appliances, Parts & Accessories, Refrigerator...","{'Material': 'Plastic', 'Color': 'Clear', 'Bra...",B0B2NP5QQT,NaN
2,Amazon Home,Ritadeshop Stove Top Burner Covers Black Marbl...,4.3,155,[Burner covers offer more counter space and co...,[Set of 2.Made of Tinplate.Size: Outer measure...,22.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Ritadeshop Large Black Marble Burn...,Ritadeshop,"[Appliances, Parts & Accessories, Range Parts ...",{'Package Dimensions': '20.63 x 12.03 x 1.19 i...,B0BKFVJ2RD,NaN
3,Tools & Home Improvement,DISNIMO Boho Style Refrigerator Door Handle Co...,4.5,204,[【Perfect Size】: The Size of one cover is 16.1...,[],11.99,[{'thumb': 'https://m.media-amazon.com/images/...,[{'title': 'Metonsto Refrigerator Handle Cover...,DISNIMO,"[Appliances, Parts & Accessories, Refrigerator...","{'Material': 'Polyester', 'Brand': 'DISNIMO', ...",B09KC11XYV,NaN
4,Appliances,"HiCOZY Dual-Mode Nugget Ice Maker Countertop, ...",4.2,201,"[Quicool Technology, Fast Ice Making: HiCOZY u...",[],399.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Returned', 'url': 'https://www.ama...",Hicozy,"[Appliances, Refrigerators, Freezers & Ice Mak...","{'Brand': 'HiCOZY', 'Model Name': 'Nugget Ice ...",B0BYZ86N13,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1509,Tools & Home Improvement,Stove Cover for Samsung Gas Range with 2pcs Ga...,3.6,121,[【Hassle-free Purchase】Please refer to the siz...,[],17.99,[{'thumb': 'https://m.media-amazon.com/images/...,[],BVSIAONG,"[Appliances, Parts & Accessories, Range Parts ...","{'Manufacturer': 'BVSIAONG', 'Part Number': 'B...",B0BMVQBK84,NaN
1510,Industrial & Scientific,"Portable Ice Maker, 26Lbs/24H Self-Cleaning Ic...",4.2,399,"[Quick and Powerful: Within only 6 minutes, yo...",[Enjoy fresh homemade ice anytime with this po...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],KINGSWERE,"[Appliances, Refrigerators, Freezers & Ice Mak...",{'Package Dimensions': '14.8 x 13.5 x 11 inche...,B09X6XVRBF,NaN
1511,Tools & Home Improvement,Sealegend 2 Pieces Dryer Vent Cleaner Kit New ...,4.2,127,[【New Upgrade】According to customer feedback o...,[],13.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],Sealegend,"[Appliances, Parts & Accessories, Dryer Parts ...","{'Manufacturer': 'Sealegend', 'Part Number': '...",B0CFQGFBPM,NaN
1512,Industrial & Scientific,"Nugget Ice Maker Countertop, Crushed Chewable ...",4.3,1360,[FAST ＆ EFFICIENT ICE MAKER: With a water tank...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],FOOING,"[Appliances, Refrigerators, Freezers & Ice Mak...",{'Product Dimensions': '15.4 x 13.07 x 10.19 i...,B0CFLQ4FM6,NaN


In [68]:
list(df_items["features"].items())[0]

(0,
 ['【Good Fits in All Refillable Pods】K cup paper filters are compatible with all standard reusable single cup filters (not included), perfect size gives you perfect brew experience',
  '【Space for Coffee Flow Through】K cup disposable filters must be shorter than refillable pods, leave sufficient space for coffee come out. Keeps grounds out of coffee cup',
  '【Improved Coffee flavor】Remove oils and sediments, full brewing and effective filtration thinner filter papers ensure origina flavor of the coffee',
  '【Save Money and Time】Reduces k cup waste and save money, easy clean up, enjoy your favorite coffee, excellent for your personal enjoyment',
  '【No Blowouts】Last without tearing though, absorbs water so coffee liquid can flow through. Prevent coffee grinds from seeping from filter'])

In [69]:
list(df_items["images"].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/51tjs-IoCeL._AC_US75_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51tjs-IoCeL._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/717EWI9-gIL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51nCErqrMZL._AC_US75_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51nCErqrMZL._AC_.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/71KwR0X8RnL._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51U2t3qKPuL._AC_US75_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51U2t3qKPuL._AC_.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/71pv1E0ic5L._AC_SL1500_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41W0nbQ3BYL._AC_US75_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41W0nbQ3BYL._AC_.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.com/images/I/71hXD5mVgXL._AC_SL1

Preprocess titles and features

In [70]:
def preprocess_description(row):
    return f"{row["title"]}{" ".join(row["features"])}"

In [71]:
def extract_first_large_image(row):
    return row["images"][0].get("large","")

In [72]:
df_items["description"] = df_items.apply(preprocess_description, axis=1)
df_items["images"] = df_items.apply(extract_first_large_image, axis=1)

In [73]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Amazon Home,HERISUN 100 Disposable Coffee Filters for Sing...,4.4,360,[【Good Fits in All Refillable Pods】K cup paper...,HERISUN 100 Disposable Coffee Filters for Sing...,7.99,https://m.media-amazon.com/images/I/51tjs-IoCe...,[],HERISUN,"[Small Appliance Parts & Accessories, Coffee &...",{'Product Dimensions': '4.7 x 2.5 x 2.5 inches...,B0BJKNMKRP,NaN
1,Tools & Home Improvement,Large Capacity Egg Holder For Refrigerator - 3...,4.6,1285,[Auto Rolling Egg Holder: Egg container keeps ...,Large Capacity Egg Holder For Refrigerator - 3...,24.99,https://m.media-amazon.com/images/I/41PG5Y6cDM...,[{'title': 'Honest review of egg holder for fr...,TOGOO,"[Appliances, Parts & Accessories, Refrigerator...","{'Material': 'Plastic', 'Color': 'Clear', 'Bra...",B0B2NP5QQT,NaN
2,Amazon Home,Ritadeshop Stove Top Burner Covers Black Marbl...,4.3,155,[Burner covers offer more counter space and co...,Ritadeshop Stove Top Burner Covers Black Marbl...,22.99,https://m.media-amazon.com/images/I/51x+0PiNQA...,[{'title': 'Ritadeshop Large Black Marble Burn...,Ritadeshop,"[Appliances, Parts & Accessories, Range Parts ...",{'Package Dimensions': '20.63 x 12.03 x 1.19 i...,B0BKFVJ2RD,NaN
3,Tools & Home Improvement,DISNIMO Boho Style Refrigerator Door Handle Co...,4.5,204,[【Perfect Size】: The Size of one cover is 16.1...,DISNIMO Boho Style Refrigerator Door Handle Co...,11.99,https://m.media-amazon.com/images/I/3195BBr20R...,[{'title': 'Metonsto Refrigerator Handle Cover...,DISNIMO,"[Appliances, Parts & Accessories, Refrigerator...","{'Material': 'Polyester', 'Brand': 'DISNIMO', ...",B09KC11XYV,NaN
4,Appliances,"HiCOZY Dual-Mode Nugget Ice Maker Countertop, ...",4.2,201,"[Quicool Technology, Fast Ice Making: HiCOZY u...","HiCOZY Dual-Mode Nugget Ice Maker Countertop, ...",399.99,https://m.media-amazon.com/images/I/31F1km7rww...,"[{'title': 'Returned', 'url': 'https://www.ama...",Hicozy,"[Appliances, Refrigerators, Freezers & Ice Mak...","{'Brand': 'HiCOZY', 'Model Name': 'Nugget Ice ...",B0BYZ86N13,NaN


In [74]:
list(df_items["description"].items())[0]

(0,
 'HERISUN 100 Disposable Coffee Filters for Single Serve 1.0 and 2.0 Coffee Filter Paper Use with Reusable K Cup Filter Pods (Natural)【Good Fits in All Refillable Pods】K cup paper filters are compatible with all standard reusable single cup filters (not included), perfect size gives you perfect brew experience 【Space for Coffee Flow Through】K cup disposable filters must be shorter than refillable pods, leave sufficient space for coffee come out. Keeps grounds out of coffee cup 【Improved Coffee flavor】Remove oils and sediments, full brewing and effective filtration thinner filter papers ensure origina flavor of the coffee 【Save Money and Time】Reduces k cup waste and save money, easy clean up, enjoy your favorite coffee, excellent for your personal enjoyment 【No Blowouts】Last without tearing though, absorbs water so coffee liquid can flow through. Prevent coffee grinds from seeping from filter')

Sample 50 items from the dataset

In [75]:
df_sample=df_items.sample(50, random_state=42)

In [76]:
len(df_sample)

50

In [77]:
data_to_embed = df_sample[["description","images","rating_number","price","average_rating","parent_asin"]].to_dict(orient="records")

In [78]:
data_to_embed

[{'description': "Gevi Household V2.0 Countertop Nugget Ice Maker | Self Cleaning Pellet Ice Machine | Open and Pour Water Refill | Store Ice Up to 24 Hours | Stainless Steel Housing | Fits Under Wall Cabinet | White【Nugget Ice】 Also known as Pellet ice or Chewblet ice. Unlike those hard ice cubes, nugget ice is made not only to cool down drinks but also to retain their flavor and provide chewable joy. Before you have to drive to chain stores for it, but now you can have it directly from your countertop whenever you like! 【Ice Ready Always】 Max 30Lbs ice in 24 hours, 4.8Lbs ice basket, 2.8L/3quarts water reservoir, extra thick insulation, auto ice refill, ice is always ready for you. 【Ultra Quiet】 Equipped with latest high performance compressor, muted exhaust fan and high precision ice making components, every effort is made to lower the noise and create an ultra quiet world of ice. 【Sleek Design】 Stainless steel housing, touch control panel, ice access from the front with Anti Drippi

In [98]:
co = cohere.ClientV2(api_key=os.getenv("COHERE_API_KEY"))
text_inputs = [
    {
        "content": [
            {"type": "text", "text": "hello"},
            {"type": "text", "text": "goodbye"}
        ]
    },
]
response = co.embed(
    inputs=text_inputs,
    model="embed-v4.0",
    input_type="classification",
    output_dimension=1536,
    embedding_types=["float"],
)
print(response)
print(response.embeddings )
print(response.embeddings.float[0])


response_type='embeddings_by_type' id='edc7ced4-2f84-4aea-a969-ec2e76a5deaa' embeddings=EmbedByTypeResponseEmbeddings(float_=[[0.026357077, -0.042990185, 0.0069411234, -0.0057256278, 0.018296419, 0.002319039, -0.035825156, 0.02712476, -0.033522107, 0.010875493, -0.0079646995, 0.05629667, -0.047084488, -0.048108064, 0.009915891, 0.011323308, -0.0078047663, 0.041710712, -0.00067172165, -0.03301032, -0.025717342, 0.023670191, 0.0029747672, -0.036592837, -0.021239199, -0.0034225818, 0.021878934, 0.0058215875, 0.033266213, -0.024309926, -0.033522107, 0.01394622, 0.009212183, 0.014969796, -0.024949662, 0.0054377466, -0.01740079, -0.01631324, 0.040175352, 0.015289664, -0.009084236, 0.008028673, 0.0039183763, -0.0047660246, 0.022646615, 0.026996814, 0.010747546, -0.016889, -0.0011195361, -0.0074209245, -0.0012954632, 0.016761053, -0.013690327, 0.0020791383, 0.03863999, 0.0007037084, 0.022262774, 0.05680846, -0.0055656936, -0.0012554798, -0.012986618, 0.019703835, -0.013882247, -0.0017832611, 0

In [99]:
def generate_embedding(text):
    response = co.embed(
        model="embed-v4.0",
        inputs=[
            {
                "content": [
                    {
                        "type": "text",
                        "text": text
                    }
                ]
            }
        ],
        input_type="classification",
        output_dimension=1536,
        embedding_types=["float"],
    )

    return response.embeddings.float[0]

In [100]:
qdrant_client = QdrantClient(url="http://localhost:6333")

g:\agentic-engineering\ai-engineering\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.13.6. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


In [101]:
qdrant_client.create_collection(
    collection_name="Amazon-items-collection-01",
    vectors_config=VectorParams(size=1536,distance=Distance.COSINE)
)

True

Embed data

In [91]:
pointStruct = PointStruct(
    id=0,
    vector=generate_embedding("Test text"),
    payload={
        "text":"test text",
        "model":"embed-v4.0"
    }
)

In [92]:
pointStruct

PointStruct(id=0, vector=[0.007233407, -0.0027256315, 0.0061501428, -0.015934462, -0.00020747997, -0.021106172, -0.06038322, 0.03913727, -0.03158937, -0.005346431, -0.0077226227, 0.035363324, -0.020407291, -0.025858555, -0.006744191, 0.0021490557, 0.015165693, 0.012579838, 0.0033895674, -0.025159676, 0.03298713, 0.009434878, -0.0129292775, -0.0075129587, -0.0064296946, 0.0035468154, 0.0112519665, 0.0087359985, 0.03438489, 0.0047873273, -0.045287415, -0.03466444, -0.027955195, 0.019289086, -0.006604415, 0.042491898, -0.0048222714, -0.026976764, 0.008107007, 0.028374523, 0.035503097, 0.034943994, -0.018869756, -0.03913727, 0.018450428, -0.016843006, 0.03885772, -0.0077575664, -0.05591039, 0.014327037, 0.023761915, 0.019568635, -0.038019065, -0.055071734, -0.015375358, -0.008701054, -0.005171711, -0.0010526879, 0.0048222714, 0.020547068, -0.016563453, -0.015165693, 0.0012929278, -0.0120906215, 0.020686844, 0.076597236, -0.026836988, 0.031030266, -0.0180311, 0.0108326385, -0.030051835, 0.0

Amazon data embedding

In [102]:
point_structs = []
for i, data in enumerate(data_to_embed):
    vector_embedding = generate_embedding(data["description"])
    point_struct = PointStruct(
        id=i,
        vector=vector_embedding,
        payload=data

    )
    point_structs.append(point_struct)

In [103]:
point_structs

[PointStruct(id=0, vector=[-0.005190131, 0.031140786, 0.0033573657, 0.019203484, 0.0147269955, -0.026729172, -0.04437562, -0.029713498, -0.033995356, -0.042040057, 0.00674717, 0.09861249, -0.029194485, 0.005027939, 0.01790595, 0.025172133, 0.02698868, 0.03451437, 0.030362265, -0.012326561, 0.049306244, 0.008823222, -0.0049306243, -0.0005919993, -0.0032276125, 0.039964005, 0.00012215054, -0.0024004355, -0.0016543542, -0.03607141, -0.00040750633, -0.010899275, -0.06591466, -0.009017852, -0.0141431065, -0.0047684326, 0.049046736, -0.03892598, 0.0010785741, 0.044116113, -0.01946299, -0.0045413645, -0.038406968, 0.02218781, 0.00327627, 0.009212482, 0.018684471, -0.024134109, 0.0061632805, 0.039444994, -0.0358119, -0.004281858, -0.02815646, -0.0027085994, 0.022706822, -0.004573803, 0.0026923805, -0.029973004, 0.025561394, 0.0358119, -0.0008271771, 0.0114831645, -0.05475588, -0.008563716, -0.06850973, 0.016997678, 0.035292886, 0.028805226, 0.03451437, -0.0020922713, -0.010315385, 0.032438315,

In [95]:
len(point_structs)

50

Writing embeddings to Qdrant

In [104]:
qdrant_client.upsert(
    collection_name="Amazon-items-collection-01",
    wait=True,
    points=point_structs
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [105]:
def retrieve_data(query,k=5):
    query_embedding = generate_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    return results

Testing Retrival

In [106]:
retrieve_data("What kind of refrigerators do you offer",k=10).points

[ScoredPoint(id=40, version=0, score=0.3624581, payload={'description': 'Classic Retro 3.5 Cu.Ft. Chest Freezer and Refrigerator All in One, Includes Rolling Wheels, Portable, Indoor/Outdoor Use, Lock and Keys, Removable Basket, Adjustable Temperature Dial with GaugeRETRO DESIGN: Its chrome accents and retro design details harkens back to the cool vibes of record spinning jukebox and shiny muscle car eras. FROM FRIDGE TO FREEZER: Versatile temperature cooling settings ranges from 45F to -20F, allowing you to either keep beverages cold, ice frozen or anything in between. SPACIOUS CAPACITY: This unit has a 3.5 Cu. Ft. capacity with refrigerator & freezer capabilities, plenty of room for beverages, meats, and more. USE IN ANY SPACE: It’s a fabulous addition to any party or rec room, office, or apartment to serve the essential need of keeping water, soda, pizza, ice, or ice cream cold. LOCK IN THE CHILL: The heavy duty gasket creates a tight seal and prevents any cold air from escaping. EA